# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrishaSolanki-coder/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use Logistic Regression as the main model because the target is binary and the model is simple to interpret. It can estimate the probability that a content item is declining and can produce a ranked list for refresh review. I will also compare it with a Decision Tree and Random Forest to check whether added model complexity improves Precision@50.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

url = "https://raw.githubusercontent.com/PrishaSolanki-coder/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

categorical_features = [
    "content_type",
    "main_intent",
    "provider_used",
    "model_used"
]

print("Dataset shape:", df.shape)
print("Target rate:", round(df["is_declining_label"].mean() * 100, 2), "%")
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


Dataset shape: (30000, 45)
Target rate: 54.21 %
Numeric features: 10
Categorical features: 4


## 2. Split design
I will use a client-level holdout split. About 20% of clients will be kept completely out of training and used for validation. This avoids putting pages from the same client in both training and validation, which gives a more honest test of whether the model can rank pages for clients it did not train on.

In [2]:
# Select clients first, then split the rows by client
clients = df["client_id"].dropna().unique()

train_clients, val_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[df["client_id"].isin(train_clients)].copy()
val_df = df[df["client_id"].isin(val_clients)].copy()

print("Total clients:", len(clients))
print("Training clients:", len(train_clients))
print("Validation clients:", len(val_clients))

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(val_df))

print("\nClient overlap:",
      len(set(train_df["client_id"]) & set(val_df["client_id"])))

print("\nTraining decline rate:",
      round(train_df["is_declining_label"].mean() * 100, 2), "%")

print("Validation decline rate:",
      round(val_df["is_declining_label"].mean() * 100, 2), "%")


Total clients: 32
Training clients: 25
Validation clients: 7

Training rows: 26581
Validation rows: 3419

Client overlap: 0

Training decline rate: 54.44 %
Validation decline rate: 52.38 %


## 3. Train + compare vs my baseline

I will compare the Week-4 baseline with Logistic Regression, Decision Tree, and Random Forest using the same validation clients and Precision@50. Precision@50 measures how many of the top 50 ranked pages are actually declining. The baseline is calculated from the same observable signals used in ML-07, while the learned models use the allowed feature set.

In [3]:
from sklearn.metrics import roc_auc_score

# -----------------------------
# 1. Recreate the Week-4 baseline
# -----------------------------

df["baseline_score"] = (
    df["impressions_90d"].rank(pct=True) * 0.5
    + (1 - df["ctr"].rank(pct=True)) * 0.3
    + (1 - df["sessions_90d"].rank(pct=True)) * 0.2
)

# Evaluate baseline only on validation clients
baseline_val = df[df["client_id"].isin(val_clients)].copy()


# -----------------------------
# 2. Precision@K function
# -----------------------------

def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["y_true"].mean()


# -----------------------------
# 3. Prepare model data
# -----------------------------

X_train = train_df[numeric_features + categorical_features].copy()
y_train = train_df["is_declining_label"].copy()

X_val = val_df[numeric_features + categorical_features].copy()
y_val = val_df["is_declining_label"].copy()


# -----------------------------
# 4. Preprocessing
# -----------------------------

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


# -----------------------------
# 5. Models
# -----------------------------

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=20,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )
}


# -----------------------------
# 6. Baseline Precision@50
# -----------------------------

baseline_p50 = precision_at_k(
    baseline_val["is_declining_label"],
    baseline_val["baseline_score"],
    k=50
)

results = [
    {
        "Method": "Week-4 Baseline",
        "Precision@50": baseline_p50
    }
]


# -----------------------------
# 7. Train and evaluate models
# -----------------------------

trained_models = {}
validation_predictions = {}

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    probabilities = pipeline.predict_proba(X_val)[:, 1]

    p50 = precision_at_k(
        y_val,
        probabilities,
        k=50
    )

    auc = roc_auc_score(
        y_val,
        probabilities
    )

    results.append({
        "Method": name,
        "Precision@50": p50,
        "ROC-AUC": auc
    })

    trained_models[name] = pipeline
    validation_predictions[name] = probabilities


# -----------------------------
# 8. Show comparison table
# -----------------------------

results_df = pd.DataFrame(results)

results_df["Precision@50"] = results_df["Precision@50"].round(3)

if "ROC-AUC" in results_df.columns:
    results_df["ROC-AUC"] = results_df["ROC-AUC"].round(3)

print(results_df.to_string(index=False))


             Method  Precision@50  ROC-AUC
    Week-4 Baseline          0.48      NaN
Logistic Regression          0.54    0.453
      Decision Tree          0.84    0.625
      Random Forest          0.54    0.651


## 4. Errors and interpretation

The model can make two main types of errors: false positives, where a page is predicted as declining but is not labeled as declining, and false negatives, where a declining page receives a lower score. I will inspect the highest-scored validation pages and compare their predicted probability with the actual label.

For interpretation, I will also check which numeric features are most associated with the Logistic Regression predictions. The model should be treated as decision-support for prioritizing pages, not as proof that a page must be refreshed.

In [4]:
# Select the Logistic Regression model
logistic_model = trained_models["Logistic Regression"]

logistic_scores = validation_predictions["Logistic Regression"]

error_df = val_df[
    ["content_id", "client_id", "is_declining_label"]
].copy()

error_df["predicted_probability"] = logistic_scores

# Classify using 0.50 probability threshold
error_df["predicted_label"] = (
    error_df["predicted_probability"] >= 0.50
).astype(int)

error_df["error_type"] = np.select(
    [
        (error_df["predicted_label"] == 1) &
        (error_df["is_declining_label"] == 0),

        (error_df["predicted_label"] == 0) &
        (error_df["is_declining_label"] == 1)
    ],
    [
        "False Positive",
        "False Negative"
    ],
    default="Correct"
)

print("Error counts:")
print(error_df["error_type"].value_counts())

print("\nTop 10 predicted declining pages:")
print(
    error_df
    .sort_values("predicted_probability", ascending=False)
    .head(10)
    .to_string(index=False)
)


# -----------------------------
# Logistic Regression feature effects
# -----------------------------

feature_names = logistic_model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = logistic_model.named_steps[
    "model"
].coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

print("\nTop model signals:")
print(
    feature_importance
    .sort_values("absolute_coefficient", ascending=False)
    .head(10)[
        ["feature", "coefficient"]
    ]
    .to_string(index=False)
)

Error counts:
error_type
Correct           1800
False Positive    1287
False Negative     332
Name: count, dtype: int64

Top 10 predicted declining pages:
          content_id         client_id  is_declining_label  predicted_probability  predicted_label     error_type
content_5c7a9bd6cbfe client_8527a891e2                   0               0.770406                1 False Positive
content_84d12054c0c0 client_9400f1b21c                   1               0.769322                1        Correct
content_38bab00f71e2 client_8527a891e2                   0               0.767657                1 False Positive
content_f488400fca67 client_9400f1b21c                   1               0.764163                1        Correct
content_df1fa766cac2 client_9400f1b21c                   1               0.763731                1        Correct
content_47757540a16b client_8527a891e2                   0               0.760164                1 False Positive
content_48cecb04b88e client_8527a891e2         

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.